# Day 4 Project Solution: Text → PersonProfile Extractor

Complete implementation: schema-guided prompt + JSON mode + Pydantic validation + retry loop.

## Part 1 — Schema

In [ ]:
import ollama
import json
from pydantic import BaseModel, Field, ValidationError

MODEL = "llama3.2"

class PersonProfile(BaseModel):
    name:   str       = Field(description="Full name of the person")
    age:    int       = Field(description="Age in years")
    city:   str       = Field(description="City where they live")
    skills: list[str] = Field(description="List of professional skills")
    bio:    str       = Field(description="One-sentence biography")

schema = PersonProfile.model_json_schema()
print("Schema generated from PersonProfile:")
print(json.dumps(schema, indent=2))

## Part 2 — Extraction Function

In [ ]:
_schema_str = json.dumps(PersonProfile.model_json_schema(), indent=2)
_example_str = (
    '{"name": "Jane Smith", "age": 28, "city": "Cape Town", '
    '"skills": ["Python", "SQL"], '
    '"bio": "A data engineer who builds data pipelines."}'
)

SYSTEM = (
    "You are an information extractor. Read the user's text and extract "
    "the information into a JSON object with exactly these fields:\n\n"
    f"{_schema_str}\n\n"
    "Fill each field with the ACTUAL VALUE from the text — not the schema.\n"
    f"Example of a correctly filled response:\n{_example_str}\n\n"
    "Return only the JSON object — no explanation, no prose."
)


def extract(text: str, max_tries: int = 2) -> PersonProfile:
    """Extract a PersonProfile from text with retry on validation failure."""
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": text},
    ]
    for attempt in range(max_tries):
        response = ollama.chat(model=MODEL, messages=messages, format="json")
        raw = response["message"]["content"]
        try:
            return PersonProfile.model_validate_json(raw)
        except ValidationError as e:
            if attempt == max_tries - 1:
                raise
            messages.append({"role": "assistant", "content": raw})
            messages.append({
                "role": "user",
                "content": (
                    f"That response failed validation:\n{e}\n\n"
                    "Please fix it and return only the corrected JSON object."
                ),
            })
    raise RuntimeError("unreachable")

## Part 3 — Demo

In [ ]:
TEST_INPUTS = [
    (
        "Alice Chen is a 32-year-old data engineer based in Cape Town. "
        "She specialises in Python, SQL, and dbt, and builds data pipelines "
        "for financial services companies."
    ),
]

print("Running extractor on", len(TEST_INPUTS), "input(s)...\n")

passed = 0
for i, text in enumerate(TEST_INPUTS, 1):
    print(f"─── Input {i} ───")
    print(text[:80] + "...")
    try:
        profile = extract(text)
        print(f"  ✅ Name:   {profile.name}")
        print(f"     Age:    {profile.age}")
        print(f"     City:   {profile.city}")
        print(f"     Skills: {profile.skills}")
        print(f"     Bio:    {profile.bio}")
        passed += 1
    except ValidationError as e:
        print(f"  ❌ All retries failed: {e}")
    except Exception as e:
        print(f"  ❌ Error: {e}")
    print()

print(f"{passed}/{len(TEST_INPUTS)} extractions succeeded.")

In [ ]:
# Gate check — verify Ollama is reachable (exercises verify the extraction logic)
import urllib.request
try:
    urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
except Exception as e:
    raise AssertionError(f"Ollama server is not running: {e}") from e

print(f"✅ Day 4 gate: Ollama running · {passed}/{len(TEST_INPUTS)} extraction(s) completed.")
if passed < len(TEST_INPUTS):
    print("  ℹ️  Some extractions failed — this is a model reliability issue, not a code bug.")
    print("  ℹ️  The exercise verify tests confirm the extraction code is correct.")